# tnWF demo — wavefunction flows via tensor networks

A minimalistic, self-contained tour of the V-step methods released in this repo,
reproducing two figures from the paper *"Exponentially-accelerated simulations of
wavefunction flows via tensor networks."*

- **Part 1** — `Dense` vs `TCI+1TDVP` on a 3-D Gaussian mixture, compared by a
  t-SNE overlay against the target (reproduces **Fig. 5**, left / *d*=3 panel).
  Driven by the closed-form analytic GMM potential (no training).
- **Part 2** — the **V-MPS + 2TDVP** pipeline at *d*=8, σ=0.5: a velocity
  potential pre-trained as a tensor train (**MPS-V**) is fed *directly* to 2-site
  TDVP with **no runtime tensor-cross** (reproduces the 2TDVP curve of **Fig. 7**).
  Training and generation are standalone scripts (`scripts/mps_v/`); the notebook
  loads a shipped checkpoint and its generated trajectory.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

from tnwf.pipelines.run_evolution import run


# Joint t-SNE over a list of point clouds; returns one embedding per cloud.
# Fitting on the concatenation puts every cloud in a shared 2-D frame so the
# overlays are directly comparable (mirrors scripts/make_fig3_tsne_grid.py).
def tsne_embed(clouds, seed=0):
    joined = np.concatenate(clouds, axis=0)
    perplexity = min(30, max(5, joined.shape[0] // 5))
    emb = TSNE(
        n_components=2, init="pca", perplexity=perplexity,
        random_state=seed, learning_rate="auto",
    ).fit_transform(joined)
    out, i = [], 0
    for c in clouds:
        out.append(emb[i:i + c.shape[0]])
        i += c.shape[0]
    return out

## Part 1 — Dense vs TCI+1TDVP on the 3-D Gaussian mixture

We evolve the same Gaussian source into a 3-D orthogonal Gaussian-mixture target
two ways:

- **`dense`** — the exact $O(N^d)$ reference (feasible at $d=3$, $N=16$);
- **`tci_tdvp1`** — the tensor-network V-step (TT-cross MPO + one-site TDVP).

Each `run(...)` returns per-step sliced-Wasserstein (`sw`), the bond dimension
(`chi_max`), and the final sample cloud (`samples_T`) alongside a `target` draw.

In [ ]:
common = dict(dataset="gmm_3d", V_source="analytic", N=16, K=16,
              n_samples=1500, save=False)

res_dense = run(method="dense", **common)
res_tdvp1 = run(method="tci_tdvp1", method_kwargs={"D_V": 16}, **common)

print(f"Dense      final SW = {res_dense['sw'][-1]:.3f}  (exact reference)")
print(f"TCI+1TDVP  final SW = {res_tdvp1['sw'][-1]:.3f}  "
      f"(chi* = {int(res_tdvp1['chi_max'].max())})")

A **joint t-SNE overlay** of the final samples shows that both methods reproduce
the target's cluster geometry — this is the *d*=3 panel of Fig. 5.

In [ ]:
target = res_dense["target"]
emb_t, emb_d, emb_v = tsne_embed(
    [target, res_dense["samples_T"], res_tdvp1["samples_T"]], seed=0)

fig, ax = plt.subplots(figsize=(6, 6), dpi=110)
ax.scatter(emb_t[:, 0], emb_t[:, 1], s=22, alpha=0.35, c="0.6", label="Target")
ax.scatter(emb_d[:, 0], emb_d[:, 1], s=16, alpha=0.70, c="#1f77b4",
           label=f"Dense  (SW={res_dense['sw'][-1]:.2f})")
ax.scatter(emb_v[:, 0], emb_v[:, 1], s=16, alpha=0.70, c="#d62728",
           label=f"TCI+1TDVP  (SW={res_tdvp1['sw'][-1]:.2f})")
ax.set(title="3-D Gaussian mixture — final samples (t-SNE overlay)\n"
             "reproduces paper Fig. 5 (left, d=3)", xticks=[], yticks=[])
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=3,
          frameon=False, fontsize=9)
fig.subplots_adjust(left=0.04, right=0.96, top=0.88, bottom=0.10)
plt.show()

## Part 2 — V-MPS + 2TDVP at *d*=8, σ=0.5 (paper Table 2)

This is the high-dimensional showcase and the paper's **trained-V bypass**. The
velocity potential is pre-trained as a real tensor train — an **MPS-V**
(`tnwf.mps_v`) — and its cores at each time are fed *directly* to the two-site
TDVP V-step. There is **no runtime tensor-cross**: `run(method="mps_v_tdvp2")`
reads `model.get_mps_cores(t)` and hands them straight to the 2-site TDVP engine.

At $d=8$ the dense grid has $N^8$ amplitudes — out of reach — so only the
tensor-network V-step is viable. Two-site TDVP lets the wavefunction bond grow
via SVD and truncates it back, so $\chi$ adapts to the state.

We load the paper's **Table 2** configuration — 2TDVP, $N=32$, $K=160$ — whose
endpoint sliced-Wasserstein sits at the sample-size floor. The reported metric is
the *grid-unbiased* SW: WF samples land on grid nodes, so a half-cell shift
($x \to x - \tfrac{1}{2}\,\Delta x$) removes the quantization bias before the SW —
that shift is the whole difference between the raw SW ($\approx 0.12$) and the
near-floor SW ($\approx 0.036$) the paper quotes.

> **How this cell's data was produced.** Training and generation are standalone
> scripts — `scripts/mps_v/train.py` fits the MPS-V and `scripts/mps_v/generate.py`
> runs the evolution. The shipped `.npz` holds the paper's actual Table 2 run:
> per-step metrics plus a cloud of samples drawn from the saved **final** state
> $\psi_{t=1}$ (sampling an MPS is instant — no re-evolution). Regenerate from
> scratch with the command shown after the plots.

In [ ]:
from pathlib import Path

# Paper Table 2 result (2TDVP, N=32, K=160): per-step metrics + samples drawn
# from the saved final wavefunction. sw_unbiased = grid-half-cell-corrected SW.
RES = next(p for p in (Path("checkpoints/mps_v_gmm_d8_result.npz"),
                       Path("examples/checkpoints/mps_v_gmm_d8_result.npz")) if p.exists())
res2 = {k: v for k, v in np.load(RES).items()}

print(f"d=8, sigma=0.5, K={int(res2['K'])}  (V-MPS + 2TDVP, no runtime cross)")
print(f"  grid N                          = {int(res2['N'])}")
print(f"  endpoint SW (unbiased)          = {float(res2['sw_unbiased'][-1]):.4f}")
print(f"  Table 2 mean over {int(res2['n_reps'])} inits       "
      f"= {float(res2['sw_unbiased_mean']):.4f} +/- {float(res2['sw_unbiased_ci']):.4f}")
print(f"  sample-size floor               = {float(res2['sw_floor']):.4f}")
print(f"  chi* (self-limiting bond)       = {int(res2['chi'].max())}")
print(f"  wall-clock (paper run)          = {float(res2['wall_cum'][-1]) / 3600:.1f} h")

The three panels mirror Fig. 7 / Table 2: **(a)** the unbiased SW descending to the
floor, **(c)** the bond dimension self-limiting near $\chi^\star\approx 16$ (pure
two-site TDVP truncates the subspace each step), and **(d)** accuracy vs the paper
run's accumulated wall-clock.

In [ ]:
steps = np.arange(len(res2["sw_unbiased"]))
floor = float(res2["sw_floor"])
mean, ci = float(res2["sw_unbiased_mean"]), float(res2["sw_unbiased_ci"])

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), dpi=110)

axes[0].plot(steps, res2["sw_unbiased"], "-o", ms=3)
axes[0].axhline(floor, ls=":", c="0.5", lw=1, label=f"floor {floor:.3f}")
axes[0].set(xlabel="Trotter step $k$", ylabel="unbiased SW",
            title=f"(a) accuracy  (final {mean:.3f}$\\pm${ci:.3f})")
axes[0].legend(loc="upper right", fontsize=8)

axes[1].plot(steps, res2["chi"], "-s", ms=3, c="C2")
axes[1].axhline(16, ls="--", c="0.5", lw=1)
axes[1].set(xlabel="Trotter step $k$", ylabel=r"$\chi_{\max}$",
            title=r"(c) bond dimension (self-limits $\chi^\star\!\approx\!16$)")

axes[2].plot(res2["wall_cum"] / 3600.0, res2["sw_unbiased"], "-o", ms=3, c="C3")
axes[2].set(xlabel="wall-clock [h]", ylabel="unbiased SW",
            title="(d) cost vs accuracy")

fig.suptitle(r"V-MPS + 2TDVP on the $d{=}8$ GMM ($\sigma{=}0.5$, $K{=}160$) "
             "— paper Fig. 7 / Table 2", y=1.04)
plt.tight_layout()
plt.show()

A t-SNE overlay of the *d*=8 final samples (drawn from the paper's $\psi_{t=1}$)
against the target confirms the trained tensor-network preparation recovers the
16-mode mixture structure in eight dimensions:

In [ ]:
emb_t8, emb_v8 = tsne_embed([res2["target"], res2["samples_T"]], seed=0)

fig, ax = plt.subplots(figsize=(5.5, 5.5), dpi=110)
ax.scatter(emb_t8[:, 0], emb_t8[:, 1], s=20, alpha=0.35, c="0.6", label="Target")
ax.scatter(emb_v8[:, 0], emb_v8[:, 1], s=16, alpha=0.70, c="#d62728",
           label=f"V-MPS + 2TDVP  (SW={float(res2['sw_unbiased_mean']):.3f})")
ax.set(title="d=8 GMM — V-MPS + 2TDVP final state (t-SNE, K=160)",
       xticks=[], yticks=[])
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=2,
          frameon=False, fontsize=9)
fig.subplots_adjust(left=0.04, right=0.96, top=0.92, bottom=0.10)
plt.show()

To reproduce the trajectory from scratch (paper Table 2 config; hours on CPU,
faster on a GPU) — or a quick lower-`K` sanity run:

```bash
python scripts/mps_v/generate.py \
    --ckpt examples/checkpoints/mps_v_gmm_d8.pt --dataset gmm_8d \
    --K 160 --D_max 24 --out examples/checkpoints/mps_v_gmm_d8_result.npz
```

The trained MPS-V checkpoint itself is produced by `scripts/mps_v/train.py`.

### What this notebook does *not* reproduce

Two elements of paper Fig. 7 rely on code outside this minimal release:

- the **1TDVP+SE** comparison curve (one-site TDVP with demand-driven subspace
  expansion) and the **hybrid** SE→TDVP-2 schedule;
- the **SW − SW$_\mathrm{cl}$** deviation panels (b), which need the classical-flow
  reference trajectory.

Here we ship the full **V-MPS + 2TDVP** bypass — train an MPS-V with
`python -m tnwf.mps_v.train` and run it via `run(method="mps_v_tdvp2", ...)` —
plus the `dense`, `tci_tdvp1`, and `tci_tdvp2` V-steps. See the
[README](../README.md) and the paper for the full set of methods.